In [2]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv('..\\data\\processed\\pdf.csv')
df.head()

,Unnamed: 0,Age,WFH_Days_Per_Week,Gender,Education_Level,Marital_Status,Has_Children,Location_Type,Department,Job_Level,...,Quality_Score,Innovation_Score,Efficiency_Rating,Meetings_Per_Week,Commute_Time_Minutes,Job_Satisfaction,Stress_Level,Work_Life_Balance,Prod_Efficiency,Exp_Task_Rate
0,0,39,2,Female,Associate Degree,Married,Yes,Urban,Product,Mid-Level,...,58.1,52.1,72.1,4,48,55.9,6,8,3763.62,566.0
1,1,33,5,Female,Master Degree,Married,No,Urban,Customer Success,Senior,...,93.3,77.9,89.5,12,0,96.1,3,8,7294.25,283.2
2,2,40,3,Male,PhD,Single,Yes,Rural,Operations,Mid-Level,...,84.7,63.2,95.0,15,24,90.4,5,6,7809.00,245.7
3,3,48,3,Male,Bachelor Degree,Married,Yes,Urban,Finance,Manager,...,67.8,82.5,95.0,8,8,100.0,10,5,7182.00,982.8
4,4,32,5,Male,High School,Divorced,Yes,Rural,Engineering,Senior,...,86.4,67.5,95.0,10,0,100.0,3,4,9310.00,589.2


In [4]:
X = df.drop(columns=['Quality_Score'])
y = df['Quality_Score']

X.shape, y.shape

((1440, 27), (1440,))

In [5]:
cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
num_cols = X.drop(columns=cat_cols).columns.tolist()

print('Full columns', len(X.columns.tolist()))
print("Categorical columns:", len(cat_cols))
print("Numerical columns:", len(num_cols))

Full columns 27
Categorical columns: 13
Numerical columns: 14


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=X['Gender'])

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((1152, 27), (1152,), (288, 27), (288,))

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)]
)

In [8]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)   
X_train_processed.shape, X_test_processed.shape

((1152, 80), (288, 80))

In [9]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.ensemble import RandomForestRegressor,HistGradientBoostingRegressor,GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor  

from sklearn.metrics import mean_absolute_error, r2_score,root_mean_squared_error
import time

modellar ={
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'ElasticNet Regression': ElasticNet(),
    'Random Forest Regressor': RandomForestRegressor(),
    'Gradient Boosting Regressor': GradientBoostingRegressor(),
    'Hist Gradient Boosting Regressor': HistGradientBoostingRegressor(),
    'K-Nearest Neighbors Regressor': KNeighborsRegressor(),
    'Support Vector Regressor': SVR(),
    'Decision Tree Regressor': DecisionTreeRegressor(),
    'MLP Regressor': MLPRegressor(max_iter=500,random_state=42),
    
    'XGBoost Regressor': XGBRegressor(objective='reg:squarederror', eval_metric='rmse'),
    'LightGBM Regressor': LGBMRegressor(objective='regression', metric='rmse',verbose=-1),
    'CatBoost Regressor': CatBoostRegressor(verbose=0, objective='RMSE')
}

In [10]:
from sklearn.metrics import mean_absolute_percentage_error

In [11]:
natijalar = []
for model_name, model in modellar.items():
    print(f"Training {model_name}...")
    start_time = time.time()
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    end_time = time.time()
    
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    
    natijalar.append({
        'Model': model_name,
        'MAE': mae,
        'R2 Score': r2,
        'RMSE': rmse,
        'MAPE': mape,
        'Training Time (s)': end_time - start_time
    })
natijalar_df = pd.DataFrame(natijalar)

Training Linear Regression...
Training Ridge Regression...
Training Lasso Regression...
Training ElasticNet Regression...
Training Random Forest Regressor...
Training Gradient Boosting Regressor...
Training Hist Gradient Boosting Regressor...
Training K-Nearest Neighbors Regressor...
Training Support Vector Regressor...
Training Decision Tree Regressor...
Training MLP Regressor...
Training XGBoost Regressor...
Training LightGBM Regressor...
Training CatBoost Regressor...


In [12]:
natijalar_df.sort_values(by='R2 Score', ascending=False)

,Model,MAE,R2 Score,RMSE,MAPE,Training Time (s)
4,Random Forest Regressor,4.511653,0.819044,5.606677,0.055533,1.071675
1,Ridge Regression,4.461952,0.814358,5.678811,0.054601,0.002814
0,Linear Regression,4.463525,0.813736,5.688318,0.054603,0.021617
13,CatBoost Regressor,4.591917,0.813558,5.691033,0.056792,2.592169
5,Gradient Boosting Regressor,4.559769,0.813058,5.698657,0.056446,0.339648
2,Lasso Regression,4.686102,0.807159,5.787867,0.057658,0.002010
10,MLP Regressor,4.635376,0.801065,5.878609,0.057120,1.581714
6,Hist Gradient Boosting Regressor,4.736171,0.798635,5.914402,0.058514,3.232015
11,XGBoost Regressor,4.706845,0.795204,5.964584,0.058126,0.116917
12,LightGBM Regressor,4.716882,0.793381,5.991074,0.058321,0.069944


In [13]:
from sklearn.model_selection import cross_validate

model = RandomForestRegressor(random_state=42)

cv = cross_validate(model, X_train_processed, y_train, cv=5, scoring=['r2', 'neg_root_mean_squared_error','neg_mean_absolute_percentage_error'], return_train_score=True,verbose=2)
cvdf = pd.DataFrame(cv)
cvdf

[CV] END .................................................... total time=   0.7s
[CV] END .................................................... total time=   0.7s
[CV] END .................................................... total time=   0.7s
[CV] END .................................................... total time=   0.7s
[CV] END .................................................... total time=   0.7s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    4.0s finished


,fit_time,score_time,test_r2,train_r2,test_neg_root_mean_squared_error,train_neg_root_mean_squared_error,test_neg_mean_absolute_percentage_error,train_neg_mean_absolute_percentage_error
0,0.826120,0.005878,0.859738,0.975870,-5.252818,-2.144172,-0.051800,-0.021045
1,0.812442,0.000000,0.817914,0.977085,-5.903556,-2.095778,-0.057625,-0.020726
2,0.803947,0.016405,0.835790,0.976325,-5.594854,-2.130987,-0.055465,-0.021104
3,0.801475,0.016245,0.823954,0.977425,-5.537749,-2.103723,-0.053552,-0.020799
4,0.786750,0.011034,0.838241,0.976431,-5.740105,-2.108662,-0.060123,-0.020735


In [14]:
tf = cvdf['test_r2'].mean()
tg = cvdf['train_r2'].mean()

round(float((tg-tf)*100), 3)

14.15

In [16]:
check_model = RandomForestRegressor(max_depth=8, min_samples_split=5,random_state=42)

cv_check =cross_validate(check_model, X_train_processed, y_train, cv=5, scoring=['r2', 'neg_root_mean_squared_error','neg_mean_absolute_percentage_error'], return_train_score=True,verbose=2)
cv_check_df = pd.DataFrame(cv_check)
cv_check_df


[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.4s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    2.4s finished


,fit_time,score_time,test_r2,train_r2,test_neg_root_mean_squared_error,train_neg_root_mean_squared_error,test_neg_mean_absolute_percentage_error,train_neg_mean_absolute_percentage_error
0,0.496085,0.006264,0.860239,0.938653,-5.243428,-3.418826,-0.051994,-0.033884
1,0.494981,0.007000,0.819073,0.944738,-5.884740,-3.254635,-0.057454,-0.032335
2,0.463351,0.016153,0.841366,0.943801,-5.499054,-3.283220,-0.054492,-0.032769
3,0.448721,0.015941,0.825641,0.943264,-5.511145,-3.335054,-0.053286,-0.032958
4,0.493016,0.006013,0.839096,0.942356,-5.724910,-3.297712,-0.059699,-0.032759


In [17]:
model_lasso = Lasso(alpha=0.1, random_state=42)
cv_lasso = cross_validate(model_lasso, X_train_processed, y_train, cv=5, scoring=['r2', 'neg_root_mean_squared_error','neg_mean_absolute_percentage_error'], return_train_score=True,verbose=2)
cv_lasso_df = pd.DataFrame(cv_lasso)
cv_lasso_df

[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished


,fit_time,score_time,test_r2,train_r2,test_neg_root_mean_squared_error,train_neg_root_mean_squared_error,test_neg_mean_absolute_percentage_error,train_neg_mean_absolute_percentage_error
0,0.007940,0.001005,0.861292,0.846630,-5.223632,-5.405683,-0.052492,-0.053678
1,0.001997,0.001000,0.826464,0.856582,-5.763288,-5.243115,-0.056364,-0.051936
2,0.002681,0.001967,0.844778,0.849405,-5.439584,-5.374527,-0.052721,-0.053739
3,0.002011,0.000000,0.832652,0.853823,-5.399217,-5.353178,-0.052922,-0.053260
4,0.004605,0.001000,0.856783,0.847043,-5.401103,-5.371834,-0.056380,-0.052904


In [19]:
model_ridge = Ridge(alpha=0.1, random_state=42)
cv_ridge = cross_validate(model_ridge, X_train_processed, y_train, cv=5, scoring=['r2', 'neg_root_mean_squared_error','neg_mean_absolute_percentage_error'], return_train_score=True,verbose=2)
cv_ridge_df = pd.DataFrame(cv_ridge)
cv_ridge_df

[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished


,fit_time,score_time,test_r2,train_r2,test_neg_root_mean_squared_error,train_neg_root_mean_squared_error,test_neg_mean_absolute_percentage_error,train_neg_mean_absolute_percentage_error
0,0.000000,0.003656,0.856364,0.854880,-5.315611,-5.258286,-0.052503,-0.051518
1,0.001534,0.001009,0.806590,0.866201,-6.084359,-5.064233,-0.059324,-0.049642
2,0.000808,0.003228,0.837882,0.859017,-5.559106,-5.200191,-0.054576,-0.051113
3,0.000000,0.002011,0.823487,0.862242,-5.545091,-5.196732,-0.054547,-0.050898
4,0.002009,0.000000,0.851351,0.855727,-5.502581,-5.217114,-0.055517,-0.051164


In [20]:
from sklearn.svm import LinearSVR

SVM_liner = LinearSVR(random_state=42,C=0.1,epsilon=0.1)
cv_svm = cross_validate(SVM_liner, X_train_processed, y_train, cv=5, scoring=['r2', 'neg_root_mean_squared_error','neg_mean_absolute_percentage_error'], return_train_score=True,verbose=2)
cv_svm_df = pd.DataFrame(cv_svm)
cv_svm_df

[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished


,fit_time,score_time,test_r2,train_r2,test_neg_root_mean_squared_error,train_neg_root_mean_squared_error,test_neg_mean_absolute_percentage_error,train_neg_mean_absolute_percentage_error
0,0.005797,0.002015,0.770182,0.790258,-6.723779,-6.321544,-0.068527,-0.060891
1,0.002001,0.001001,0.753055,0.799872,-6.875051,-6.193566,-0.068280,-0.059918
2,0.001353,0.002172,0.767164,0.796147,-6.662144,-6.253084,-0.064807,-0.061082
3,0.002008,0.000000,0.758694,0.799000,-6.483416,-6.277254,-0.062361,-0.060790
4,0.001142,0.001009,0.791719,0.790853,-6.513436,-6.281494,-0.066293,-0.060593


In [ ]:
from sklearn.ensemble import StackingRegressor

stacking = StackingRegressor(
    estimators=[('linear', LinearRegression()), ('lasso', Lasso(alpha=0.1, random_state=42))],
    final_estimator=Ridge(alpha=0.1,random_state=42),
    cv=3,
    verbose=2)
stacking.fit(X_train_processed, y_train)
y_pred_stacking = stacking.predict(X_test_processed)


[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.0s finished


In [23]:
r2_score(y_test, y_pred_stacking)

0.8235068511262327

In [36]:
model =Lasso(alpha=0.1, random_state=42)
model.fit(X_train_processed, y_train)
y_pred = model.predict(X_test_processed)
r2_score(y_test, y_pred)

0.8237331532577685

In [71]:
model.coef_

array([ 8.91932315e-03,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        1.43594547e-01,  4.92020939e-01, -2.22472908e+00,  0.00000000e+00,
       -5.28180876e-02,  1.95856423e-01,  6.44226796e-02,  0.00000000e+00,
        1.38956829e+01,  1.83108776e-01, -0.00000000e+00,  0.00000000e+00,
        0.00000000e+00, -0.00000000e+00,  0.00000000e+00, -0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
       -0.00000000e+00, -0.00000000e+00,  4.53388867e-01, -4.98845273e-16,
        0.00000000e+00, -0.00000000e+00, -0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00, -0.00000000e+00,  0.00000000e+00,
        0.00000000e+00, -0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00, -0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00, -5.94768775e-01,
        0.00000000e+00,  0.00000000e+00, -0.00000000e+00,  0.00000000e+00,
       -0.00000000e+00,  

In [ ]:
import joblib
joblib.dump(model, '..\Remote-Work-Productivity-ML\models\Quality\Lasso_model.joblib')
joblib.dump(preprocessor, '..\Remote-Work-Productivity-ML\models\Quality\preprocessor.joblib')